# CV Pipeline — Probabilistic Power Price Forecasting

This notebook is the CV-aligned version of project two.

> Built probabilistic deep-learning models for 24-hour electricity price forecasting, comparing QR-DNN, Normal-DNN and Johnson SU-DNN approaches; evaluated predictive distributions using Pinball loss and Winkler score; produced upside/downside price bands for scenario analysis and risk-aware trading decisions.

The notebook compares QR-DNN, Normal-DNN and Johnson SU-DNN, then writes summary metrics and upside/downside price bands.

## 1. Setup

Run this notebook from inside `Electricity_Price_Load_Forecasting_3/`. If you open it from the repository root, the setup cell tries to move into the project folder automatically.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != 'Electricity_Price_Load_Forecasting_3':
    candidate = PROJECT_DIR / 'Electricity_Price_Load_Forecasting_3'
    if candidate.exists():
        PROJECT_DIR = candidate

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

import run_cv_project_pipeline as cv

PROJECT_DIR

## 2. Experiment setup

These are the three models named in the CV bullet. Set `RUN_RECALIBRATION = True` only if you want to retrain/recalibrate; otherwise the notebook loads saved recalibration results if present.

In [ ]:
TASK_NAME = 'EM_price'
HYPER_MODE = 'load_tuned'
RUN_RECALIBRATION = False

cv.EXPERIMENTS

## 3. Run model comparison

For each model, this cell loads or computes the recalibrated predictions, evaluates Pinball loss and Winkler score, and exports per-model price bands.

In [ ]:
output_dir = PROJECT_DIR / 'experiments' / 'tasks' / TASK_NAME / 'cv_project_summary'
output_dir.mkdir(parents=True, exist_ok=True)

summary_rows = []
predictions_by_model = {}
configs_by_model = {}

for model_name, exp_conf in cv.EXPERIMENTS.items():
    predictions, configs = cv.load_or_run_experiment(
        model_name=model_name,
        exp_conf=exp_conf,
        task_name=TASK_NAME,
        hyper_mode=HYPER_MODE,
        run_recalibration=RUN_RECALIBRATION,
    )
    predictions_by_model[model_name] = predictions
    configs_by_model[model_name] = configs
    summary_rows.append(
        cv.score_experiment(
            model_name=model_name,
            predictions=predictions,
            configs=configs,
            task_name=TASK_NAME,
            output_dir=output_dir,
        )
    )

summary = pd.DataFrame(summary_rows).sort_values('avg_pinball_loss')
summary.to_csv(output_dir / 'model_comparison_summary.csv', index=False)
summary

## 4. Best model and price bands

The best model is selected by average Pinball loss. The generated price bands are the practical output behind the CV phrase: upside/downside price bands for scenario analysis and risk-aware trading decisions.

In [ ]:
best_model = summary.iloc[0]['model']
best_configs = configs_by_model[best_model]
best_quantiles = cv.build_target_quantiles(best_configs['model_config']['target_alpha'])

best_bands = cv.build_price_bands(
    predictions_by_model[best_model],
    task_name=TASK_NAME,
    quantiles=best_quantiles,
)
best_bands.to_csv(output_dir / 'best_model_price_bands.csv')

print(f'Best model by average Pinball loss: {best_model}')
print(f'Outputs written to: {output_dir}')

best_bands.head()

## 5. Visual check

The chart below shows realized price, median forecast and upside/downside bands for the first five test days.

In [ ]:
import matplotlib.pyplot as plt

plot_slice = best_bands.iloc[:24 * 5]

ax = plot_slice[['realized_price', 'median_price']].plot(figsize=(14, 5))
lower_col = [c for c in plot_slice.columns if c.startswith('downside_q_')][0]
upper_col = [c for c in plot_slice.columns if c.startswith('upside_q_')][0]

ax.fill_between(
    plot_slice.index,
    plot_slice[lower_col],
    plot_slice[upper_col],
    alpha=0.2,
)
ax.set_title(f'{best_model}: realized price, median forecast and uncertainty band')
ax.set_xlabel('Datetime')
ax.set_ylabel('Electricity price')
ax.grid(True)
plt.show()

## 6. Interview explanation

A concise way to explain the project:

> I built a probabilistic day-ahead electricity price forecasting pipeline. Instead of predicting only a point estimate, I forecasted the conditional distribution of the next 24 hourly prices. I compared QR-DNN, Normal-DNN and Johnson SU-DNN. QR-DNN directly outputs quantiles and is trained with Pinball loss, while Normal-DNN and Johnson SU-DNN output distribution parameters and are trained with negative log-likelihood. I evaluated the forecasts using Pinball loss and Winkler score, then converted the predictive distributions into upside and downside price bands for scenario analysis and risk-aware trading decisions.